In [ ]:
UNI_RANDOM_SEED = 2024
DEVICE = 0

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

np.random.seed(UNI_RANDOM_SEED) 
torch.manual_seed(UNI_RANDOM_SEED)

torch.cuda.manual_seed(UNI_RANDOM_SEED)
torch.cuda.manual_seed_all(UNI_RANDOM_SEED)

torch.cuda.set_device(DEVICE)

import pdb
import pickle as pkl
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path

try:
    import open3d
    from visual_utils import open3d_vis_utils as V
    OPEN3D_FLAG = True
except:
    import mayavi.mlab as mlab
    from visual_utils import visualize_utils as V
    OPEN3D_FLAG = False

from cudaext.ops.Rotated_IoU.oriented_iou_loss import cal_iou_3d, assign_target_3d

from pytorch3d.vis.plotly_vis import plot_scene

from pcdet.datasets.kitti.kitti_dataset import create_kitti_infos
from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.datasets import KittiDataset, build_dataloader
from pcdet.models import build_network, load_data_to_gpu
from pcdet.utils import common_utils

from data_tools import adv_dataset, kitti_carla_dataset
from eval_utils import eval_utils
from loss_utils import mesh_objectwise_loss, relevant_bounding_box_loss
from optim_utils import objectwise_deepfool

EVAL_OUTPUT_DIR = "./eval_output/"
CFG_FILE = "./cfgs/kitti_models/pointrcnn.yaml"
DATA_CONFIG_FILE = "./cfgs/dataset_configs/kitti_dataset.yaml"
DATA_PATH = "/home/ksas/Public/datasets/KITTI"
CKPT_PATH = "/home/ksas/Public/model_zoo/pcdet/pointrcnn_7870.pth"
# ROOFTOP_ANNOTATE = "/home/ksas/uzuki_space/vehicle-shape-reconstruction/rooftop_appro.pkl"
ROOFTOP_ANNOTATE = None

BATCH_SIZE = 1
WORKERS = 4
DIST_TEST = False

OPTIM = "rbboxloss"
EVAL_INIT_PATH = False

cfg_from_yaml_file(CFG_FILE, cfg)

# BATCH_SIZE = cfg.OPTIMIZATION.BATCH_SIZE_PER_GPU
logger = common_utils.create_logger()
logger.info('-----------------Kitti Attack Test-------------------------')

laplacian_weights = 0.001
learning_rate = 0.005
overshoot = 0.02


In [ ]:
def roipooling_grad_mapping(pooled_features_grad, batch_point_features, pooled_pts_idx):
    
    batch_size = batch_point_features.size(0)
    npoint = batch_point_features.size(1)
    feature_size = batch_point_features.size(2)
    
    batch_point_features_grad = torch.zeros_like(batch_point_features)
    xyz_features_grad = batch_point_features.new_zeros((batch_size, npoint, 3))
    
    for batch_mask in range(0, batch_size):
        pts_idx_expanded = pooled_pts_idx[batch_mask].view(-1).long().unsqueeze(1).expand(-1, feature_size)
        xyz_pooled_features_grad_viewed = pooled_features_grad[:, :, :3].view(-1, 3)
        pooled_features_grad_viewed = pooled_features_grad[:, :, 3:].view(-1, feature_size)

        batch_point_features_grad[batch_mask].scatter_add_(0, pts_idx_expanded, pooled_features_grad_viewed)
        xyz_features_grad[batch_mask].scatter_add_(0, pts_idx_expanded[:, :3], xyz_pooled_features_grad_viewed)
    
    return xyz_features_grad, batch_point_features_grad

rooftop_approximate = None
try:
    with open(ROOFTOP_ANNOTATE, "rb") as input:
        rooftop_approximate = pkl.load(input)
except FileNotFoundError as error:
    logger.info(error.__str__())
except TypeError as error:
    logger.info(error.__str__())
    
logger.info(f"rooftop_approximate: {rooftop_approximate}")

In [ ]:
test_set, test_loader, sampler = build_dataloader(
        dataset_cfg=cfg.DATA_CONFIG,
        class_names=cfg.CLASS_NAMES,
        batch_size=BATCH_SIZE,
        dist=DIST_TEST, workers=WORKERS, logger=logger, training=False
    )
logger.info(f'Class names of samples: \t{test_set.class_names}')

In [ ]:
model = build_network(model_cfg=cfg.MODEL, num_class=len(cfg.CLASS_NAMES), dataset=test_set)
model.load_params_from_file(filename=CKPT_PATH, logger=logger, to_cpu=True)
model.cuda()
model.eval()

for idx, module in enumerate(model.module_list):
    logger.info(f'Module names of model \t({idx}): \t{module._get_name()}')
    
backbone_network = model.module_list[0]
point_headbox = model.module_list[1]
pointrcnn_head = model.module_list[2]

kitti_adv_dataset = adv_dataset(test_set,
                                sample_amount=50,
                                rooftop_approximate = rooftop_approximate,
                                surrogate_model=None)

optimizer = optim.Adam([kitti_adv_dataset.universal_adv_patch.get_mesh_deform_vert(),
                        kitti_adv_dataset.universal_adv_patch.theta,
                        kitti_adv_dataset.universal_adv_patch.global_translation], 
                       lr=learning_rate)

In [ ]:
if EVAL_INIT_PATH:
    kitti_adv_dataset.enable_adversarial_patch(True)
    eval_utils.eval_one_epoch(
            cfg, None, model, kitti_adv_dataset, 0, logger, dist_test=DIST_TEST,
            result_dir=Path(EVAL_OUTPUT_DIR)
            , infer_time=True
        )

Pure Deepfooled grad:

Car AP@0.70, 0.70, 0.70:
bbox AP:73.0101, 70.7567, 64.9182
bev  AP:71.5738, 64.6090, 62.6272
3d   AP:59.8438, 52.9202, 46.8179
aos  AP:72.34, 69.15, 63.13
Car AP_R40@0.70, 0.70, 0.70:

In [ ]:
criterion = mesh_objectwise_loss(freezed_iou = False, 
                                #  normalized = False, 
                                 verbose=False)
deepfool_perturbate = objectwise_deepfool(model=model, freezed_iou = False, 
                                 normalized = False, 
                                 verbose=False)

rbbox_loss_func = relevant_bounding_box_loss(frozen_iou = False,
                 frozen_logit = False,
                 confidence_threshold = 0.1,
                 iou_threshold = 0.1, 
                 verbose = True)

def evaluate_one_epoch_attack(enable_adv, update, visualize, verbose_epoch: int = 100):
    mesh_loss_scaler = []
    regular_loss_scaler = []
    kitti_adv_dataset.enable_adversarial_patch(enable_adv)
    
    for i, batch_dict in tqdm(enumerate(kitti_adv_dataset), total=kitti_adv_dataset.__len__()):
        # batch_dict = kitti_adv_dataset.__getitem__(1)
        load_data_to_gpu(batch_dict)

        model.eval()
        model.zero_grad()
        pred_dicts, _ = model(batch_dict)
        point_headbox_ret_dict = point_headbox.forward_ret_dict
        pointrcnn_head_ret_dict = pointrcnn_head.forward_ret_dict

        if not torch.eq(batch_dict['gt_boxes'][0, :, 7], 1).any():
            # logger.info(f"no vehicles found in batch \t{i}")
            continue
            
        if OPTIM == "meshloss":
            mesh_loss = criterion(batch_dict = point_headbox_ret_dict, 
                                    point_coords = batch_dict["point_coords"][:, 1:4].squeeze(dim=0),
                                    gt_boxes = batch_dict["gt_boxes"], 
                                    target_class = 1,
                                    ret_part_loss = False)
            regular_loss = kitti_adv_dataset.universal_adv_patch.get_laplacian_loss()
            total_loss = mesh_loss + laplacian_weights * regular_loss
            
            optimizer.zero_grad()
            total_loss.backward()
            
            mesh_loss_scaler.append(mesh_loss.item())
            regular_loss_scaler.append(regular_loss.item())
            
            assert not torch.isnan(mesh_loss), "mesh loss is NaN"
            
        elif OPTIM == "deepfool":
            deepfooled_grad, iou_grad = deepfool_perturbate(batch_dict = point_headbox_ret_dict, 
                    point_coords = batch_dict["point_coords"][:, 1:4].squeeze(dim=0),
                    deform_vert = kitti_adv_dataset.universal_adv_patch.get_mesh_deform_vert(),
                    gt_boxes = batch_dict["gt_boxes"], 
                    target_class = 1)
            model.zero_grad()
            if kitti_adv_dataset.universal_adv_patch.get_mesh_gradient() is not None:
                kitti_adv_dataset.universal_adv_patch.get_mesh_gradient().zero_()
            
            regular_loss = kitti_adv_dataset.universal_adv_patch.get_laplacian_loss()
            regular_loss.backward()
            
            regular_grad = kitti_adv_dataset.universal_adv_patch.get_mesh_deform_vert().grad.detach().clone()
            kitti_adv_dataset.universal_adv_patch.get_mesh_deform_vert().grad = -(1+overshoot) * deepfooled_grad
        elif OPTIM == "rbboxloss":
            mesh_loss = rbbox_loss_func(batch_dict = point_headbox_ret_dict, 
                                    gt_boxes = batch_dict["gt_boxes"], 
                                    target_class = 1,
                                    logit_normal = "sigmoid",
                                    ret_part_loss = False)
            
            regular_loss = kitti_adv_dataset.universal_adv_patch.get_laplacian_loss()
            total_loss = mesh_loss + laplacian_weights * regular_loss
            
            optimizer.zero_grad()
            total_loss.backward()
        else:
            raise NotImplementedError
        
        if verbose_epoch > 0 and i % verbose_epoch == 0:
            logger.info(f"deformed verts of mesh: \t{kitti_adv_dataset.universal_adv_patch.get_mesh_deform_vert()}")
            logger.info(f"deformed verts of mesh: \t{kitti_adv_dataset.universal_adv_patch.get_mesh_gradient()}")
            # print(deepfool_perturbate.debug_msg)
            if visualize:
                fig = plot_scene({
                    "original": {
                        "mesh_1": kitti_adv_dataset.universal_adv_patch.get_basic_mesh()
                    },
                    "adversarial": {
                        "mesh_1": kitti_adv_dataset.universal_adv_patch.get_deformed_mesh()
                    },
                }, ncols=2)
                fig.update_layout(height=400, width=800)
                fig.show()
                
                V.draw_scenes(
                    points=batch_dict['points'][:, 1:], ref_boxes=pred_dicts[0]['pred_boxes'].detach(),
                    ref_scores=pred_dicts[0]['pred_scores'].detach(), ref_labels=pred_dicts[0]['pred_labels'].detach(), gt_boxes=batch_dict['gt_boxes'][0]
                )
            
        if update:
            """
                set grad along z-axi to 0.
            """
            vert_grad, translate_grad, theta_grad = kitti_adv_dataset.universal_adv_patch.get_mesh_gradient()
            vert_grad[:, 2] = 0.
            translate_grad[2] = 0.
            
            if OPTIM == "meshloss":
                optimizer.step()
            elif OPTIM == "rbboxloss":
                optimizer.step()
            elif OPTIM == "deepfool":
                new_vert = kitti_adv_dataset.universal_adv_patch.get_mesh_deform_vert() - kitti_adv_dataset.universal_adv_patch.get_mesh_gradient()
                kitti_adv_dataset.universal_adv_patch.update_mesh(new_vert)
            else:
                raise NotImplementedError
            
            # """
            #     limit distortion into [-0.1m, 0.1m]
            # """
            # vert = kitti_adv_dataset.universal_adv_patch.get_mesh_deform_vert()
            # with torch.no_grad():
            #     vert[:] = vert.clamp(min=-0.1, max=0.1)
            
evaluate_one_epoch_attack(enable_adv = True, 
                            update = True, 
                            visualize = True, 
                            verbose_epoch= -1)

In [ ]:
fig = plot_scene({
    "original": {
        "mesh_1": kitti_adv_dataset.universal_adv_patch.get_basic_mesh()
    },
    "adversarial": {
        "mesh_1": kitti_adv_dataset.universal_adv_patch.get_deformed_mesh()
    },
    
}, ncols=2)
fig.update_layout(height=400, width=800)
fig.show()

logger.info(f"deformed verts of mesh: \t{kitti_adv_dataset.universal_adv_patch.get_mesh_deform_vert()}")
logger.info(f"theta of mesh: \t{kitti_adv_dataset.universal_adv_patch.theta}")
logger.info(f"theta of global_translation: \t{kitti_adv_dataset.universal_adv_patch.global_translation}")


# plt.plot(np.arange(mesh_loss_scaler.__len__()), mesh_loss_scaler, label='Mesh loss')

# # # 绘制第二条曲线
# # plt.plot(x_values, y2_values, label='Curve 2')

# # 添加图例
# plt.legend()

# # 添加标题和坐标轴标签
# plt.title('Mesh loss')
# plt.xlabel('X Axis')
# plt.ylabel('Y Axis')

# # 显示图形
# plt.show()

# plt.plot(np.arange(regular_loss_scaler.__len__()), regular_loss_scaler, label='Laplacian loss')

# # # 绘制第二条曲线
# # plt.plot(x_values, y2_values, label='Curve 2')

# # 添加图例
# plt.legend()

# # 添加标题和坐标轴标签
# plt.title('Laplacian loss')
# plt.xlabel('X Axis')
# plt.ylabel('Y Axis')

# # 显示图形
# plt.show()
# print(kitti_adv_dataset.universal_adv_patch.get_mesh_deform_vert())

In [ ]:
kitti_adv_dataset.enable_adversarial_patch(True)
eval_utils.eval_one_epoch(
        cfg, None, model, kitti_adv_dataset, 0, logger, dist_test=DIST_TEST,
        result_dir=Path(EVAL_OUTPUT_DIR)
        , infer_time=True
    )

Clean
Car AP@0.70, 0.70, 0.70:
bbox AP:98.0106, 90.4862, 90.3078
bev  AP:90.3662, 88.9447, 88.5946
3d   AP:89.2901, 79.2153, 78.7551
aos  AP:97.96, 90.39, 90.13

Patch added
Car AP@0.70, 0.70, 0.70:
bbox AP:81.2036, 74.7322, 67.8265
bev  AP:79.6103, 72.2210, 66.4427
3d   AP:67.8708, 60.0782, 54.4692
aos  AP:80.31, 73.21, 66.26

Patch added and optimized
Car AP@0.70, 0.70, 0.70:
bbox AP:79.0949, 72.7988, 66.5548
bev  AP:73.6473, 70.6503, 64.5577
3d   AP:65.1627, 54.5988, 51.9095
aos  AP:78.27, 71.39, 65.01

Patch added (scaled)
Car AP@0.70, 0.70, 0.70:
bbox AP:73.0291, 70.8378, 65.0790
bev  AP:71.2238, 64.7269, 63.0242
3d   AP:59.4674, 52.8792, 50.8481
aos  AP:72.07, 69.02, 63.02

Patch added and optimized (scaled)
Car AP@0.70, 0.70, 0.70:
bbox AP:69.1211, 64.1674, 62.4920
bev  AP:67.5980, 62.2795, 56.4149
3d   AP:55.7378, 49.6803, 44.7865
aos  AP:67.81, 62.25, 59.91



ar AP@0.70, 0.70, 0.70:
bbox AP:73.0291, 70.8378, 65.0790
bev  AP:71.2238, 64.7269, 63.0242
3d   AP:59.4674, 52.8792, 50.8481
aos  AP:72.07, 69.02, 63.02
Car AP_R40@0.70, 0.70, 0.70:
bbox AP:75.6936, 69.9060, 64.4972
bev  AP:72.5187, 66.5600, 62.6646
3d   AP:60.4770, 52.1773, 48.1110
aos  AP:74.53, 68.08, 62.32
Car AP@0.70, 0.50, 0.50:
bbox AP:73.0291, 70.8378, 65.0790
bev  AP:73.7594, 72.1752, 66.2935
3d   AP:73.4797, 71.7306, 65.9993
aos  AP:72.07, 69.02, 63.02
Car AP_R40@0.70, 0.50, 0.50:
bbox AP:75.6936, 69.9060, 64.4972
bev  AP:76.6063, 72.5528, 68.7663
3d   AP:76.2505, 72.1809, 66.8703
aos  AP:74.53, 68.08, 62.32



Car AP_R40@0.70, 0.70, 0.70:
bbox AP:69.2459, 65.9491, 60.6775
bev  AP:67.6649, 62.6394, 57.5929
3d   AP:54.6303, 48.0308, 43.2368
aos  AP:67.86, 63.66, 58.08



Car AP@0.70, 0.50, 0.50:
bbox AP:69.1211, 64.1674, 62.4920
bev  AP:70.1725, 69.6444, 64.0626
3d   AP:69.8799, 69.2944, 63.5847
aos  AP:67.81, 62.25, 59.91
Car AP_R40@0.70, 0.50, 0.50:
bbox AP:69.2459, 65.9491, 60.6775
bev  AP:71.3599, 68.5271, 64.7946
3d   AP:71.0045, 68.1349, 62.9652
aos  AP:67.86, 63.66, 58.08


Car AP_R40@0.70, 0.70, 0.70:
bbox AP:82.6373, 76.3993, 70.4407
bev  AP:79.9304, 73.0078, 67.2479
3d   AP:68.0128, 58.7108, 52.8206
aos  AP:81.68, 74.69, 68.51
Car AP@0.70, 0.50, 0.50:
bbox AP:81.2036, 74.7322, 67.8265
bev  AP:81.9749, 75.8437, 73.9331
3d   AP:81.8360, 75.6438, 73.7331
aos  AP:80.31, 73.21, 66.26
Car AP_R40@0.70, 0.50, 0.50:
bbox AP:82.6373, 76.3993, 70.4407
bev  AP:83.4352, 79.0004, 73.1383
3d   AP:83.2994, 78.7224, 72.8745
aos  AP:81.68, 74.69, 68.51



Car AP_R40@0.70, 0.70, 0.70:
bbox AP:79.3155, 73.3447, 67.4530
bev  AP:76.2684, 69.7591, 64.1040
3d   AP:64.4351, 55.1361, 49.6153
aos  AP:78.49, 71.82, 65.70
Car AP@0.70, 0.50, 0.50:
bbox AP:79.0949, 72.7988, 66.5548
bev  AP:79.9024, 74.3323, 72.7876
3d   AP:79.7262, 74.0543, 67.3099
aos  AP:78.27, 71.39, 65.01
Car AP_R40@0.70, 0.50, 0.50:
bbox AP:79.3155, 73.3447, 67.4530
bev  AP:80.0880, 76.0127, 71.6473
3d   AP:79.9196, 75.7292, 69.9077
aos  AP:78.49, 71.82, 65.70